# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import duckdb
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
tok = None
for env_candidate in ['.env', '../.env', '../../.env', '../../../.env']:
    if Path(env_candidate).exists():
        with open(env_candidate) as f:
            for line in f:
                if line.startswith('HF_TOKEN='):
                    tok = line.split('=',1)[1].strip()
                    print(f'Token loaded from {env_candidate}')
                    break
        if tok:
            break
if not tok:
    tok = os.environ.get('HF_TOKEN')
    if tok:
        print('Token loaded from environment')
    else:
        print('ERROR: HF_TOKEN not found in .env or environment')
con = duckdb.connect()
con.execute('SET enable_progress_bar = false')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{tok}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

Token loaded from ../../.env


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My verdict system
- CONFIRMED - The signal checks out as expected
- OPPOSITE - The signal is backwards (e.g., fresh when it should be stale)
- MIXED - Sometimes yes, sometimes no
- FALSE - This signal doesn't apply (e.g., no data)

### Two signals:
1. Staleness - How long has the page been since it was last updated? (Check days_since_last_update)
2. Low Click-Through Rate (CTR) - Are people clicking when they see this page in search results? (Check ctr and avg_position)

A page is worth reviewing if its staleness is long (high days_since_last_update) and it has low CTR relative to expectation.

### The scoring formula
- `visibility` = the logarithmic scale of impressions (`log(impressions_90d)`)
- `is_stale` = is the page stale (0 or 1 based on days_since_last_update larger than 180)
- `is_low_ctr` = 1 and 0 for the page has low ctr or not.

##### Compute the baseline_refresh_score
`baseline_refresh_score = visibility * is_state * is_low_ctr`

### Reason Codes:
- `stale_and_low_ctr_visible` - Old page with lower CTR than expected that still gets views
- `stale_visible_page` - Old page that still gets views
- `low_ctr_visible_page` - People see it but don't click as much as expected
- `other` - Catch-all for remaining cases
- `no_issue` - Scored 0, no refresh needed

In [2]:
import pandas as pd

# Use relative path to avoid leaking local absolute paths (e.g., /Users/jasonpham/...)
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} rows from starter dataset")

Loaded 30,000 rows from starter dataset


In [3]:
df

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.00,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.00,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.00,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.00,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.00,good,page_3_5,down,-34.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,content_c322796023c8,client_e29c9c180c,10.0,0.05,LOW,0.00,keyword article,transactional,1386.0,9084.0,...,8000-15000,0.00,0.0,0.00,0.00,0.00,low,top_3,new,NaN
29996,content_526572edb3fa,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2654.0,17056.0,...,15000-25000,0.39,6.6,0.00,66.67,0.00,moderate,page_1,down,-75.1
29997,content_38112bdd0c6e,client_349c41201b,10.0,1.00,HIGH,0.00,keyword article,transactional,2857.0,18725.0,...,15000-25000,0.19,4.1,0.00,0.00,0.00,good,page_1,down,-66.2
29998,content_ab26273a7e7a,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,transactional,NaN,NaN,...,NaN,0.22,6.0,1.73,4.06,0.00,excellent,page_1,down,-27.9


In [4]:
print("Staleness:", len(df[df['days_since_last_update'] >= 180]))


Staleness: 174


In [5]:
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Signal 1: Staleness (freshness_tier)
print("--- Signal 1: Staleness ---")
bucket_1 = df.groupby('freshness_tier').agg(
    n=('content_id', 'count'),
    decline_rate=('is_declining_label', 'mean')
).sort_index()
print(bucket_1)
print("\nVerdict: MIXED - Staleness alone doesn't show a perfectly clean stair-step up in decline rate.")

# Signal 2: CTR vs Position
print("\n--- Signal 2: Low CTR on Page 1 ---")
page_1 = df[df['position_tier'] == 'page_1'].copy()
page_1['ctr_bucket'] = pd.qcut(page_1['ctr'], q=5, duplicates='drop')
bucket_2 = page_1.groupby('ctr_bucket').agg(
    n=('content_id', 'count'),
    decline_rate=('is_declining_label', 'mean')
).sort_index()
print(bucket_2)
print("\nVerdict: CONFIRMED - For a given position tier (page 1), lower CTR correlates with much higher decline rates.")

--- Signal 1: Staleness ---
                    n  decline_rate
freshness_tier                     
0-30            20480      0.511377
181+              174      0.471264
31-90             175      0.588571
91-180           9171      0.611057

Verdict: MIXED - Staleness alone doesn't show a perfectly clean stair-step up in decline rate.

--- Signal 2: Low CTR on Page 1 ---
                   n  decline_rate
ctr_bucket                        
(-0.001, 0.08]  4762      0.597228
(0.08, 0.23]    2341      0.604870
(0.23, 0.5]     2352      0.565051
(0.5, 100.0]    2359      0.483680

Verdict: CONFIRMED - For a given position tier (page 1), lower CTR correlates with much higher decline rates.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# Rule: We want to review pages that:
# 1. Have had some impressions (to make sure it's valid).
# 2. Have high staleness (days_since_last_update is high)
# 3. Have low CTR relative to others

import numpy as np
import os

df['visibility'] = np.log1p(df['impressions_90d'])
df['has_impressions'] = (df['impressions_90d'] > 10).astype(int)
df['is_stale'] = (df['days_since_last_update']>=180).astype(int)
df['is_low_ctr'] = ((df['avg_position']<=20) & (df['avg_position']>0)  & (df['ctr']<3.0)).astype(int)

# Compute baseline score
df['baseline_refresh_score'] = df['visibility']* df['is_stale'] * df['is_low_ctr']

def assign_reason(row):
    if row['baseline_refresh_score'] == 0:
        return 'no_issue'

    # Check the exact boolean flags instead of continuous factors
    if row['is_stale'] == 1 and row['is_low_ctr'] == 1:
        return 'stale_and_low_ctr_visible'
    elif row['is_stale'] == 1:
        return 'stale_visible_page'
    elif row['is_low_ctr'] == 1:
        return 'low_ctr_visible_page'

    return 'other'

df['reason_code'] = df.apply(assign_reason, axis=1)
df['action'] = np.where(df['baseline_refresh_score'] > 0, 'Review & Refresh', 'Leave as is')

queue = df[['content_id', 'client_id', 'baseline_refresh_score', 'reason_code', 'action']].sort_values('baseline_refresh_score', ascending=False)
os.makedirs('../outputs', exist_ok=True)
queue.to_csv('../outputs/baseline_action_score.csv', index=False)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(f"Base rate (is_declining_label): {df['is_declining_label'].mean():.3f}")
print(f"Precision@50: {precision_at_k(df['baseline_refresh_score'], df['is_declining_label'], 50):.3f}")
print(f"Precision@100: {precision_at_k(df['baseline_refresh_score'], df['is_declining_label'], 100):.3f}")


Base rate (is_declining_label): 0.542
Precision@50: 0.780
Precision@100: 0.600


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [7]:
top_20 = df.sort_values('baseline_refresh_score', ascending=False).head(20)
top_20

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,visibility,has_impressions,is_stale,is_low_ctr,baseline_refresh_score,reason_code,action
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.0,LOW,0.00,keyword article,informational,5125.0,33705.0,...,down,-85.6,1,11.029699,1,1,1,11.029699,stale_and_low_ctr_visible,Review & Refresh
21268,content_0a91db491d14,client_7f2253d7e2,0.0,0.0,LOW,0.00,keyword article,informational,3478.0,21948.0,...,down,-51.8,1,9.495519,1,1,1,9.495519,stale_and_low_ctr_visible,Review & Refresh
12045,content_c2d929d83eaa,client_7f2253d7e2,0.0,0.0,LOW,0.00,keyword article,informational,4758.0,30070.0,...,down,-62.8,1,8.930494,1,1,1,8.930494,stale_and_low_ctr_visible,Review & Refresh
5327,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.0,LOW,0.00,keyword article,informational,3388.0,21742.0,...,down,-52.2,1,8.424420,1,1,1,8.424420,stale_and_low_ctr_visible,Review & Refresh
20837,content_928af3e22c80,client_7f2253d7e2,0.0,0.0,LOW,0.00,keyword article,informational,3118.0,20396.0,...,down,-45.7,1,7.437206,1,1,1,7.437206,stale_and_low_ctr_visible,Review & Refresh
22872,content_e3ff1b093148,client_d029fa3a95,0.0,0.0,LOW,0.00,keyword article,informational,4758.0,33575.0,...,down,-68.5,1,7.250636,1,1,1,7.250636,stale_and_low_ctr_visible,Review & Refresh
26840,content_7f116ae1f6f5,client_9400f1b21c,NaN,NaN,NaN,NaN,keyword article,NaN,1335.0,9375.0,...,down,-44.5,1,6.861711,1,1,1,6.861711,stale_and_low_ctr_visible,Review & Refresh
26799,content_77d4d5930e5e,client_7f2253d7e2,0.0,0.0,LOW,0.00,keyword article,informational,4020.0,26513.0,...,down,-55.0,1,6.720220,1,1,1,6.720220,stale_and_low_ctr_visible,Review & Refresh
7452,content_72496874f806,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1504.0,10770.0,...,down,-22.4,1,6.711740,1,1,1,6.711740,stale_and_low_ctr_visible,Review & Refresh
11630,content_6226ee6adc91,client_d029fa3a95,0.0,0.0,LOW,0.00,keyword article,informational,3950.0,27607.0,...,down,-28.5,1,6.302619,1,1,1,6.302619,stale_and_low_ctr_visible,Review & Refresh


### Top 20 Hand Review

My evaluation:

1. **content_cf56e2e2e282**
   - **Action:** Review & Refresh
   - **Reason:** stale_and_low_ctr_visible
   - **Confidence:** Low
   - **Why wrong?** The average position is 19.7 (bottom of page 2). A CTR of 0.15% at position 20 is actually quite normal, not necessarily "low".

2. **content_0a91db491d14**
   - **Action:** Review & Refresh
   - **Reason:** stale_and_low_ctr_visible
   - **Confidence:** High
   - **Why wrong?** Position 10.5 (bottom of page 1) with 0.49% CTR. Might be normal if the SERP is crowded with answer boxes for this informational query.

3. **content_c2d929d83eaa**
   - **Action:** Review & Refresh
   - **Reason:** stale_and_low_ctr_visible
   - **Confidence:** Low
   - **Why wrong?** Position 17.9 naturally has low CTR (page 2).

4. **content_fe16a55cd13d**
   - **Action:** Review & Refresh
   - **Reason:** stale_and_low_ctr_visible
   - **Confidence:** Low
   - **Why wrong?** Position 16.4 naturally has low CTR.

5. **content_928af3e22c80**
   - **Action:** Review & Refresh
   - **Reason:** stale_and_low_ctr_visible
   - **Confidence:** Low
   - **Why wrong?** Position 15.8 naturally has low CTR.

6. **content_e3ff1b093148**
   - **Action:** Review & Refresh
   - **Reason:** stale_and_low_ctr_visible
   - **Confidence:** High
   - **Why wrong?** Position 7.8 with 0.28% CTR on an informational page. Might be a highly competitive head term where SERP features (like answer boxes) steal clicks.

7. **content_7f116ae1f6f5**
   - **Action:** Review & Refresh
   - **Reason:** stale_and_low_ctr_visible
   - **Confidence:** Medium
   - **Why wrong?** Missing intent and low word count (1335). Might be a thin navigational page that we don't care about refreshing.

8. **content_77d4d5930e5e**
   - **Action:** Review & Refresh
   - **Reason:** stale_and_low_ctr_visible
   - **Confidence:** Low
   - **Why wrong?** Position 18.6 naturally has low CTR.

9. **content_72496874f806**
   - **Action:** Review & Refresh
   - **Reason:** stale_and_low_ctr_visible
   - **Confidence:** Medium
   - **Why wrong?** Position 5.8 with 0.24% CTR. This is very low for top 6. But missing intent and 1504 words could mean it's a navigational page for someone else's brand.

10. **content_6226ee6adc91**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** Low
    - **Why wrong?** Position 17.8 naturally has low CTR.

11. **content_fd16e3475c29**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** High
    - **Why wrong?** Position 9.0 with 0.0% CTR! Might be ranking for irrelevant terms that users immediately ignore.

12. **content_b65fe2792b44**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** Low
    - **Why wrong?** Position 16.7 naturally has low CTR.

13. **content_4f241bad48a3**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** Low
    - **Why wrong?** Position 19.1 naturally has low CTR. It's a commercial page so maybe there are heavy ads above it.

14. **content_3c770ef0121a**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** Medium
    - **Why wrong?** Position 6.9 with 0.75% CTR. This is low, but not as egregious. Could be an answer-box heavy query.

15. **content_ea41fe5cf292**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** High
    - **Why wrong?** Position 8.0 with 0.0% CTR. 4000+ words. Ranking but totally ignored by users.

16. **content_df1fa766cac2**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** Medium
    - **Why wrong?** Position 4.8 with 0.49% CTR. Top 5 position should have much higher CTR. Might be a branded term for someone else.

17. **content_958a46db26bd**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** High
    - **Why wrong?** Position 6.7 with 0.0% CTR. Highly suspicious.

18. **content_02b0d6e30129**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** Medium
    - **Why wrong?** Transactional intent with 0.0% CTR at position 6.9. Missing word count. Maybe it's a broken page or highly competitive e-commerce term.

19. **content_f488400fca67**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** High
    - **Why wrong?** Position 5.7 with 0.0% CTR. Missing intent. Should definitely be investigated.

20. **content_180fabd3a3e0**
    - **Action:** Review & Refresh
    - **Reason:** stale_and_low_ctr_visible
    - **Confidence:** Medium
    - **Why wrong?** Position 6.0 with 0.83% CTR. Below average for top 6, but could be a query with strong intent for another site.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks
The main weakness in this rule is the **flat CTR threshold**. The rule checks `ctr < 3.0` for all pages ranking between 1 and 20. However, the expected CTR curve is exponential: a CTR of 2.0% at position 2 is terrible, but a CTR of 1.0% at position 15 is completely normal. 
This causes the rule to over-flag pages ranking on page 2 (positions 11-20), as seen in Picks #1, #3, #4, #5, #8, #10, #12, and #13. A better baseline might use position-specific CTR thresholds (e.g., `< 5%` for Top 3, `< 1%` for Page 2).

### Leakage Check
- **No target leakage:** The score strictly uses `impressions_90d`, `days_since_last_update`, `avg_position`, and `ctr`. It completely avoids the `trend_direction` and `trend_pct` columns.
- **No future windows:** All features are based on the trailing 90-day window or historical content properties. There are no `_last_30d` metrics mixed incorrectly.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.